In [0]:
%run ../init_framework_silver

In [0]:
# 1. SOURCE INGESTION
# Read from Bronze CDF. 
df_bronze_customers = read_delta_stream(
    spark=spark, 
    table_name=CUSTOMERS_BRONZE
)

In [0]:
# 2. SCHEMA ALIGNMENT
CUSTOMER_RENAME_MAP = {
    "annual_inc": "annual_income",
    "addr_state": "address_state",
    "zip_code": "address_zipcode",
    "country": "address_country",
    "tot_hi_cred_lim": "total_high_credit_limit", 
    "annual_inc_joint": "joint_annual_income"
}

# Standardize column names to match target Silver DDL
df_renamed_customers = standardize_column_names(df_bronze_customers, CUSTOMER_RENAME_MAP, strict=True)


In [0]:
# 4. METADATA ENRICHMENT
# Injects standard audit columns (_ingested_at, etc.)
df_with_metadata = apply_silver_metadata(df_renamed_customers)

In [0]:
# 5. DATA QUALITY: CRITICAL NULLS
# Drop records missing annual income; critical dependency for downstream risk modeling.
col_ls = ["annual_income"]
df_valid_customers = drop_critical_nulls(df_with_metadata, col_ls)


In [0]:
# 6. TRANSFORMATION: EMPLOYMENT DURATION
from pyspark.sql.functions import col, regexp_replace

# Strip non-digits and cast to INT (e.g., "10+ years" -> 10)
df_emplength_cleaned = df_valid_customers.withColumn(
    "emp_length", regexp_replace(col("emp_length"), r"\D+", "").cast("int")
)


In [0]:
# 7. NULL HANDLING: DEFAULT VALUES
# Default missing employment duration to -1 for aggregation safety.
df_emplength_notnull = df_emplength_cleaned.fillna(-1, subset=["emp_length"])


In [0]:
# 8. TRANSFORMATION: ADDRESS CLEANING
from pyspark.sql.functions import lit, length, trim, when, upper

# Enforce strict 2-char state codes; fallback to 'NA' for invalid strings.
df_customers_final = df_emplength_notnull.withColumn(
    "address_state",
    when(
        length(trim(col("address_state"))) == 2, 
        upper(trim(col("address_state")))
    )
    .otherwise(lit("NA")) 
)

In [0]:
# --- 9. UPSERT LOGIC & STREAM EXECUTION ---
# Configured for 8-core cluster (16 shuffle partitions).
spark.conf.set("spark.sql.shuffle.partitions", "16")

def upsert_customers_to_silver(microBatchDF, batchId):
    """
    Stateless foreachBatch merge. 
    Includes version fencing on _bronze_version to prevent late-arriving data overlaps.
    """
    # Import spark session
    spark_session = microBatchDF.sparkSession
    
    from pyspark.sql import Window
    from pyspark.sql.functions import col, row_number, desc
    
    # Filter for valid CDC events (insert or post-image)
    df_updated = microBatchDF = microBatchDF.filter(col("_change_type").isin("insert", "update_postimage"))

    # 1. Define the Window
    # Partition by the Primary Key and order by timestamp descending.
    # This ensures the freshest CDC event always receives rank 1.
    window = Window.partitionBy("member_id").orderBy(col("_updated_at").desc())

    # 2. Add the rank column ONCE
    # This creates a single "Parent" plan for both branches
    df_ranked = df_updated.withColumn("rn", row_number().over(window))

    # --- Branch Logic ---
    # 3. Clean: The absolute latest truth (rn=1) for the Silver Layer MERGE
    df_clean = df_ranked.filter(col("rn") == 1).drop("rn")

    # 4. Bad/Stale: Older intermediate updates (rn>1) for the Audit/Rejected table
    df_bad = df_ranked.filter(col("rn") > 1).drop("rn")

    # --- EXECUTE YOUR WRITES HERE ---    
    # 5. Register Clean Source View for SQL transformations
    df_clean.createOrReplaceTempView("customers_batch_updates")

    # 6. Execute Idempotent Merge into Silver
    merge_query = f"""
        MERGE INTO {CUSTOMERS_SILVER} AS target
        USING customers_batch_updates AS source
        ON target.member_id = source.member_id
        WHEN MATCHED AND source._bronze_version > target._bronze_version THEN
          UPDATE SET
            target.emp_title = source.emp_title,
            target.emp_length = source.emp_length,
            target.home_ownership = source.home_ownership,
            target.annual_income = source.annual_income,
            target.address_state = source.address_state,
            target.address_zipcode = source.address_zipcode,
            target.address_country = source.address_country,
            target.grade = source.grade,
            target.sub_grade = source.sub_grade,
            target.verification_status = source.verification_status,
            target.total_high_credit_limit = source.total_high_credit_limit,
            target.application_type = source.application_type,
            target.joint_annual_income = source.joint_annual_income,
            target.verification_status_joint = source.verification_status_joint,
            target._ingested_at = source._ingested_at,
            target._source_file = source._source_file,
            target._bronze_version = source._bronze_version,
            target._updated_at = source._updated_at,
            target._batch_id = source._batch_id
        WHEN NOT MATCHED THEN
          INSERT (
            member_id, emp_title, emp_length, home_ownership, annual_income, 
            address_state, address_zipcode, address_country, grade, sub_grade, 
            verification_status, total_high_credit_limit, application_type, 
            joint_annual_income, verification_status_joint, 
            _ingested_at, _source_file, _bronze_version, _updated_at, _batch_id
          )
          VALUES (
            source.member_id, source.emp_title, source.emp_length, source.home_ownership, source.annual_income, 
            source.address_state, source.address_zipcode, source.address_country, source.grade, source.sub_grade, 
            source.verification_status, source.total_high_credit_limit, source.application_type, 
            source.joint_annual_income, source.verification_status_joint, 
            source._ingested_at, source._source_file, source._bronze_version, source._updated_at, _batch_id
          )
    """
    spark_session.sql(merge_query)


# --- STREAM TRIGGER ---
# Execute trigger-once pipeline for batch-like efficiency on streaming logic.
query_customers = (
    df_customers_final.writeStream
    .outputMode("append") 
    .option("checkpointLocation", SILVER_CHECKPOINT_CUSTOMERS)
    .trigger(availableNow=True)
    .foreachBatch(upsert_customers_to_silver)
    .start()
)

# Block notebook termination until micro-batch is committed
query_customers.awaitTermination()

In [0]:
%sql
select count(1) from lending_risk.silver.customers
-- 3334
-- 6673
-- 10006